# NYC Yellow Taxi Trip-Duration Prediction
**Negar Kamvij | CS672 Project 2 | Fall 2026**

## Project objective
> Build a Deep Learning model, TensorFlow framework, (based on Neural Networks, not on classic Machine Learning models) that provides reliable and improved accuracy of a NYC Yellow Taxi trip/ride duration.

This project combines NYC Yellow Taxi records and NYC weather data, compares the required TensorFlow regression models, selects the best configuration, and reviews its predictions. The extra-credit section compares TensorFlow and PyTorch DNNs.

## 1. Setup

In [ ]:
from __future__ import annotations
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import urlencode
from datetime import datetime, timezone
import importlib.util
import subprocess
import sys
import os
import json
import hashlib
import shutil
import time

packages = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow",
            "matplotlib": "matplotlib", "sklearn": "scikit-learn",
            "tensorflow": "tensorflow", "torch": "torch", "joblib": "joblib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import torch
import joblib
from IPython.display import display, Image

ROOT = Path.cwd()
for folder in ["data", "outputs", "reports"]:
    (ROOT / folder).mkdir(exist_ok=True)
SEED = 672
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
WEATHER = ["temperature_2m", "relative_humidity_2m", "precipitation", "rain",
           "snowfall", "wind_speed_10m", "surface_pressure"]
NUMERIC = ["passenger_count", "log_trip_distance", "days_since_year_start"]
CATEGORICAL = ["pickup_zone", "destination_zone", "pickup_hour", "pickup_weekday", "pickup_month"]
TARGET = "duration_minutes"

## 2. Data preprocessing and weather integration
The modeling data are a fixed-seed sample of **50,000 eligible trips from January–November 2020 NYC TLC Yellow Taxi records**, combined with hourly NYC weather. The weather source is ERA5 through Open-Meteo, an alternative allowed by the assignment. Temperatures are Celsius. Rain and total precipitation are in mm; new snowfall is in cm.

Trip duration is the difference between drop-off and pickup times in minutes. Taxi timestamps are converted from New York local time to UTC, then matched to the latest whole-hour weather timestamp. Weather precipitation covers the preceding hour. Original files are retained. Records with invalid timestamps, nonpositive durations, or pickups outside their source month are not eligible for the modeling sample. Long trips are flagged rather than automatically deleted.

Recorded trip distance is transformed with `log1p` before scaling. Negative or missing distances are marked unavailable and imputed later using training data. This is offline regression using recorded distance; a live pre-trip application would require an estimated route distance. ERA5 is retrospective weather, not a live forecast.

Sources: [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) and [Open-Meteo historical weather](https://open-meteo.com/en/docs/historical-weather-api).

In [ ]:
def file_hash(path: Path) -> str:
    """Return a file's SHA-256 digest without loading the whole file into memory."""
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()



def download_file(url: str, path: Path) -> Path:
    """Download a missing file atomically; preserve existing local source files."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        partial = Path(str(path)+'.part')
        for attempt in range(3):
            try:
                with urlopen(Request(url, headers={'User-Agent':'CS672-taxi-study'}), timeout=120) as response, partial.open('wb') as output:
                    shutil.copyfileobj(response, output)
                partial.replace(path)
                break
            except OSError:
                if attempt == 2:
                    raise
                time.sleep(2)
    return path



def weather_classes(frame: pd.DataFrame) -> pd.Series:
    """Return exclusive weather labels, keeping missing precipitation unknown."""
    missing=frame[['rain','snowfall','precipitation']].isna().any(axis=1)
    rain=frame.rain.gt(0);snow=frame.snowfall.gt(0)
    return pd.Series(np.select([missing,rain&snow,snow,rain,frame.precipitation.eq(0)],
                              ['unknown','mixed','snow','rain','dry'],default='other_or_trace'),index=frame.index)



def load_weather(root: Path) -> pd.DataFrame:
    """Download/cache full-year ERA5 and return hourly UTC values with provenance.

    Precipitation describes the preceding hour. Reanalysis is retrospective and
    does not establish the real-time availability of these weather measurements.
    """
    path = root/'data/weather_era5_2020.json'
    params = dict(latitude=40.7075, longitude=-74.0113, start_date='2019-12-31', end_date='2021-01-02',
                  hourly=','.join(WEATHER), models='era5', timezone='UTC', temperature_unit='celsius',
                  wind_speed_unit='kmh', precipitation_unit='mm')
    url = 'https://archive-api.open-meteo.com/v1/archive?'+urlencode(params)
    download_file(url,path)
    payload=json.loads(path.read_text())
    units={'temperature_2m':'°C','relative_humidity_2m':'%','precipitation':'mm','rain':'mm','snowfall':'cm','wind_speed_10m':'km/h','surface_pressure':'hPa'}
    assert all(payload['hourly_units'][k]==v for k,v in units.items())
    weather=pd.DataFrame(payload['hourly'])
    weather['weather_time_utc']=pd.to_datetime(weather.pop('time'),utc=True)
    weather=weather.sort_values('weather_time_utc').reset_index(drop=True)
    assert weather.weather_time_utc.is_unique
    assert weather.weather_time_utc.diff().dropna().eq(pd.Timedelta(hours=1)).all()
    weather['weather_class']=weather_classes(weather)
    provenance=dict(url=url,dataset='ERA5 reanalysis via Open-Meteo',requested_location=[40.7075,-74.0113],
                    returned_grid_location=[payload['latitude'],payload['longitude']],units=units,
                    downloaded_at_or_first_recorded_utc=datetime.now(timezone.utc).isoformat(),
                    limitation='Retrospective reconstruction, not an as-issued forecast or live-feed backtest',sha256=file_hash(path))
    report=root/'reports';report.mkdir(exist_ok=True)
    if not (report/'weather_provenance_2020.json').exists():
        (report/'weather_provenance_2020.json').write_text(json.dumps(provenance,indent=2))
    local=weather.weather_time_utc.dt.tz_convert('America/New_York')
    coverage=weather.loc[local.dt.year.eq(2020)].copy()
    coverage['month']=local.loc[coverage.index].dt.month
    coverage.groupby(['month','weather_class']).size().rename('distinct_hours').reset_index().to_csv(report/'weather_coverage_2020.csv',index=False)
    return weather



def local_utc(values: pd.Series) -> pd.Series:
    """Convert naive New York timestamps to UTC; flag unresolved DST times as NaT."""
    return pd.to_datetime(values,errors='coerce').dt.tz_localize('America/New_York',ambiguous='NaT',nonexistent='NaT').dt.tz_convert('UTC')



def prepare_sample(root: Path, weather: pd.DataFrame, sample_size: int = 50000) -> pd.DataFrame:
    """Audit every monthly source row and uniformly sample eligible trips.

    January-November form one uniform-priority modeling sample. Original files stay intact. Unknown zones remain categories.
    Eligibility requires valid UTC times, positive duration and intended month;
    long rides are flagged but not discarded by a duration or distance threshold. Recorded distance is used in this offline regression; live use would require an estimated route distance.
    """
    report=root/'reports';report.mkdir(exist_ok=True)
    pool=pd.DataFrame();test=pd.DataFrame();audits=[];manifest=[]
    cols=['tpep_pickup_datetime','tpep_dropoff_datetime','PULocationID','DOLocationID','passenger_count','trip_distance']
    for month in range(1,12):
        name=f'yellow_tripdata_2020-{month:02d}.parquet'
        path=download_file('https://d37ci6vzurychx.cloudfront.net/trip-data/'+name,root/'data'/name)
        raw=pd.read_parquet(path,columns=cols)
        pickup=local_utc(raw.tpep_pickup_datetime);dropoff=local_utc(raw.tpep_dropoff_datetime)
        duration=(dropoff-pickup).dt.total_seconds()/60
        local=pickup.dt.tz_convert('America/New_York')
        in_month=local.dt.year.eq(2020)&local.dt.month.eq(month)
        valid=pickup.notna()&dropoff.notna()&np.isfinite(duration)&duration.gt(0)&in_month
        audit={'month':month,'source_rows':len(raw),'eligible_rows':int(valid.sum()),'ineligible_not_deleted':int((~valid).sum()),
               'missing_or_unresolved_time':int((pickup.isna()|dropoff.isna()).sum()),'outside_intended_month':int((~in_month).sum()),
               'nonpositive_or_unavailable_duration':int((~np.isfinite(duration)|duration.le(0)).sum()),
               'long_duration_flag':int(duration.gt(180).sum()),'negative_or_nonfinite_distance':int((raw.trip_distance.lt(0)|~np.isfinite(raw.trip_distance)).sum()),
               'long_distance_flag':int(raw.trip_distance.gt(100).sum()),'source_rows_removed':0}
        audits.append(audit)
        # Random priorities do not depend on trip duration or weather.
        priority=np.random.default_rng(SEED+month).random(len(raw))
        indices=np.flatnonzero(valid.to_numpy());cap=sample_size
        if len(indices)>cap:
            indices=indices[np.argpartition(priority[indices],cap-1)[:cap]]
        candidate=pd.DataFrame({'row_id':[f'{month:02d}:{i}' for i in indices],
                                'pickup_utc':pickup.iloc[indices].to_numpy(),'dropoff_utc':dropoff.iloc[indices].to_numpy(),
                                TARGET:duration.iloc[indices].to_numpy(),'priority':priority[indices],
                                'passenger_count':raw.passenger_count.iloc[indices].to_numpy(),
                                'trip_distance':raw.trip_distance.iloc[indices].to_numpy()})
        for source,dest in [('PULocationID','pickup_zone'),('DOLocationID','destination_zone')]:
            zones=raw[source].iloc[indices]
            candidate[dest]=zones.where(zones.between(1,263)&zones.mod(1).eq(0),0).fillna(0).astype(int).astype(str).to_numpy()
        candidate['passenger_count']=candidate.passenger_count.where(candidate.passenger_count.ge(0))
        candidate['log_trip_distance']=np.log1p(candidate.trip_distance.where(candidate.trip_distance.ge(0)&np.isfinite(candidate.trip_distance)))
        pool=pd.concat([pool,candidate],ignore_index=True).nsmallest(sample_size,'priority')
        manifest.append({'month':month,'file':name,'sha256':file_hash(path),'source_rows':len(raw),'eligible_rows':int(valid.sum())})
        print(f'Month {month:02d}: {len(raw):,} source rows, {int(valid.sum()):,} eligible; source rows removed: 0',flush=True)
        del raw,pickup,dropoff,duration,priority,local
    frame=pd.concat([pool,test],ignore_index=True).sort_values('pickup_utc').reset_index(drop=True)
    frame['pickup_utc']=pd.to_datetime(frame.pickup_utc,utc=True)
    frame['dropoff_utc']=pd.to_datetime(frame.dropoff_utc,utc=True)
    assert frame.row_id.is_unique
    frame['weather_time_utc']=frame.pickup_utc.dt.floor('h')
    frame=frame.merge(weather,on='weather_time_utc',how='left',validate='many_to_one')
    assert len(frame)==len(pool)+len(test)
    # A join timestamp is a measurement time, not a proven publication time.
    assert frame.weather_time_utc.le(frame.pickup_utc).all()
    frame['weather_class']=frame.weather_class.fillna('unmatched')
    local=frame.pickup_utc.dt.tz_convert('America/New_York')
    frame['pickup_hour']=local.dt.hour.astype(str)
    frame['pickup_weekday']=local.dt.dayofweek.astype(str)
    frame['pickup_month']=local.dt.month.astype(str)
    frame['days_since_year_start']=(local.dt.tz_localize(None)-pd.Timestamp('2020-01-01')).dt.total_seconds()/86400
    pd.DataFrame(audits).to_csv(report/'source_audit_2020.csv',index=False)
    (report/'data_manifest_2020.json').write_text(json.dumps({'seed':SEED,'sample_size':len(pool),'modeling_period':'January-November 2020','sampling':'Uniform random priorities over eligible source rows, not equal monthly quotas','files':manifest},indent=2))
    (root/'outputs').mkdir(exist_ok=True)
    frame.to_parquet(root/'outputs/assignment_sample.parquet',index=False)
    return frame

In [ ]:
weather = load_weather(ROOT)
sample_path = ROOT / "outputs/assignment_sample.parquet"
if sample_path.exists():
    data = pd.read_parquet(sample_path)
else:
    data = prepare_sample(ROOT, weather, sample_size=50000)
assert len(data) == 50000
assert data["pickup_utc"].dt.tz_convert("America/New_York").dt.month.lt(12).all()
display(pd.read_csv(ROOT / "reports/source_audit_2020.csv"))
display(data[["pickup_utc", "trip_distance", "passenger_count", "pickup_zone",
              "destination_zone", "temperature_2m", "rain", "snowfall", "duration_minutes"]].head())
display(data[NUMERIC + WEATHER + [TARGET]].isna().sum().rename("missing_values").to_frame())

## 3. Training/validation split and feature scaling
Trips are sorted by pickup time and divided chronologically into approximately **80% training and 20% validation**. The boundary is aligned to a whole hour so the same weather timestamp is not shared between partitions. Two trips crossing the boundary are excluded from the training view. The final counts are **39,999 training** and **9,999 validation** records, or **80.0012% / 19.9988%**.

Numeric inputs are imputed using training medians and independently scaled to **0–1** using a training-fitted MinMaxScaler. Validation values outside the training range are clipped to that range. Pickup/destination zones and calendar categories are one-hot encoded. All model inputs are numeric; duration remains in minutes.

In [ ]:
def split_assignment(frame: pd.DataFrame, root: Path) -> dict[str,pd.DataFrame]:
    """Split January-November trips chronologically, targeting 80%/20% without future labels.

    Whole-hour grouping prevents shared weather timestamps across partitions.
    Rides crossing the validation boundary remain in the source audit but are
    excluded from the training view. Exact realized counts are reported.
    """
    frame=frame.sort_values('pickup_utc')
    boundary=frame.pickup_utc.iloc[int(len(frame)*.8)].ceil('h')
    before=frame.loc[frame.pickup_utc.lt(boundary)]
    train=before.loc[before.dropoff_utc.lt(boundary)].copy()
    validation=frame.loc[frame.pickup_utc.ge(boundary)].copy()
    assert train.dropoff_utc.max()<validation.pickup_utc.min()
    assert set(train.weather_time_utc).isdisjoint(validation.weather_time_utc)
    result={'modeling_period':'January-November 2020','target_ratio':'80/20',
            'validation_start_local':str(boundary.tz_convert('America/New_York')),
            'train':len(train),'validation':len(validation),'boundary_overlap_excluded_from_training_view':len(before)-len(train),
            'training_percent':100*len(train)/(len(train)+len(validation)),'source_rows_deleted':0}
    (root/'reports').mkdir(exist_ok=True)
    (root/'reports/split_assignment.json').write_text(json.dumps(result,indent=2))
    return {'train':train,'validation':validation}



def prepare_inputs(splits: dict[str,pd.DataFrame], with_weather: bool = True) -> tuple[object,dict[str,np.ndarray],dict[str,np.ndarray]]:
    """Fit numeric imputation/scaling and category encoding on training rows only."""
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import MinMaxScaler,OneHotEncoder
    numeric=NUMERIC+(WEATHER if with_weather else [])
    # Fixed calendar/zone domains are public definitions, not validation-derived vocabularies.
    categories=[[str(i) for i in range(264)],[str(i) for i in range(264)],
                [str(i) for i in range(24)],[str(i) for i in range(7)],[str(i) for i in range(1,13)]]
    preprocessing=ColumnTransformer([
        ('numeric',Pipeline([('missing',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scale',MinMaxScaler(feature_range=(0,1),clip=True))]),numeric),
        ('category',OneHotEncoder(categories=categories,handle_unknown='ignore',sparse_output=False,dtype=np.float32),CATEGORICAL),
    ],sparse_threshold=0)
    preprocessing.fit(splits['train'])
    inputs={name:preprocessing.transform(part).astype('float32') for name,part in splits.items()}
    targets={name:part[TARGET].to_numpy(dtype='float32') for name,part in splits.items()}
    assert all(np.isfinite(x).all() for x in inputs.values())
    return preprocessing,inputs,targets

In [ ]:
splits = split_assignment(data, ROOT)
preprocessor, inputs, targets = prepare_inputs(splits, with_weather=True)
joblib.dump(preprocessor, ROOT / "outputs/preprocessor.joblib")
data_fingerprint = file_hash(sample_path)
display(pd.Series(json.loads((ROOT / "reports/split_assignment.json").read_text()), name="split").to_frame())
display(pd.DataFrame({name: {"rows": len(targets[name]), "input_columns": inputs[name].shape[1],
                            "minimum_input": float(inputs[name].min()),
                            "maximum_input": float(inputs[name].max())}
                      for name in ["train", "validation"]}).T)

## 4. TensorFlow regression models
The required models are:

| Model | Hidden layers | Output |
| --- | --- | --- |
| MLP | 64 ReLU units | One linear duration output |
| Linear regression | None | One linear duration output |
| DNN | 64 and 32 ReLU units | One linear duration output |

All three are implemented with TensorFlow/Keras. The experiment grid uses **MSE and MAE losses**, **SGD, Adam and RMSProp optimizers**, and learning rates **0.001 and 0.01**. Each completed run trains for **100 epochs** with batch size **32**. A validation-MAE checkpoint is retained while all 100 epochs are completed.

In [ ]:
def metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str,float]:
    """Return common duration metrics in minutes without silently clipping predictions."""
    from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
    actual=np.asarray(actual).reshape(-1);predicted=np.asarray(predicted).reshape(-1)
    return {'mae_minutes':float(mean_absolute_error(actual,predicted)),
            'rmse_minutes':float(np.sqrt(mean_squared_error(actual,predicted))),
            'r2':float(r2_score(actual,predicted)),
            'p90_absolute_error_minutes':float(np.quantile(np.abs(actual-predicted),.9)),
            'negative_prediction_count':int((predicted<0).sum())}



def experiment_grid() -> list[dict]:
    """Return the explicit 36-run grid covering three architectures and required settings."""
    import itertools
    return [dict(architecture=a,loss=l,optimizer=o,learning_rate=lr,with_weather=True,epochs=100,batch_size=32,seed=SEED)
            for a,l,o,lr in itertools.product(['linear','mlp','dnn'],['mse','mae'],['sgd','adam','rmsprop'],[.001,.01])]



def build_tensorflow(config: dict, width: int) -> object:
    """Build a Keras linear, one-hidden-layer MLP, or two-hidden-layer DNN."""
    import tensorflow as tf
    tf.keras.utils.set_random_seed(config['seed'])
    layers=[tf.keras.layers.Input(shape=(width,))]
    if config['architecture'] in ['mlp','dnn']:
        layers.append(tf.keras.layers.Dense(64,activation='relu'))
    if config['architecture']=='dnn':
        layers.append(tf.keras.layers.Dense(32,activation='relu'))
    layers.append(tf.keras.layers.Dense(1))
    model=tf.keras.Sequential(layers)
    optimizer={'sgd':tf.keras.optimizers.SGD,'adam':tf.keras.optimizers.Adam,'rmsprop':tf.keras.optimizers.RMSprop}[config['optimizer']](learning_rate=config['learning_rate'])
    model.compile(optimizer=optimizer,loss=config['loss'],metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')])
    return model



def fit_tensorflow(config: dict, inputs: dict[str,np.ndarray], targets: dict[str,np.ndarray], root: Path, fingerprint: str) -> dict:
    """Train a 100-epoch run, checkpoint by validation MAE and save all loss curves.

    A numerically failed run is recorded as failed, never as a completed run.
    The test arrays are not accessed. Existing successful runs can be resumed
    only when the data fingerprint and complete configuration agree.
    """
    import tensorflow as tf
    import matplotlib.pyplot as plt
    assert config['epochs']==100 and config['batch_size']==32
    import inspect
    definition_hash=hashlib.sha256((inspect.getsource(build_tensorflow)+inspect.getsource(prepare_inputs)).encode()).hexdigest()
    identity=hashlib.sha256(json.dumps({'config':config,'data':fingerprint},sort_keys=True).encode()).hexdigest()[:12]
    run_id=f"{config['architecture']}_{config['loss']}_{config['optimizer']}_{config['learning_rate']}_{identity}"
    folder=root/'outputs/runs'/run_id;folder.mkdir(parents=True,exist_ok=True)
    result_path=folder/'result.json'
    if result_path.exists():
        existing=json.loads(result_path.read_text())
        if existing.get('status')=='complete' and existing.get('definition_hash')==definition_hash:return existing
    config={k:(v.item() if isinstance(v,np.generic) else v) for k,v in config.items()}
    result={'run_id':run_id,**config,'hardware':('GPU' if tf.config.list_physical_devices('GPU') else 'CPU'),'data_fingerprint':fingerprint,'definition_hash':definition_hash,'status':'running'}
    result_path.write_text(json.dumps(result,indent=2))
    start=time.perf_counter();history=None
    try:
        tf.keras.backend.clear_session()
        model=build_tensorflow(config,inputs['train'].shape[1])
        options=tf.data.Options();options.threading.private_threadpool_size=1
        training=tf.data.Dataset.from_tensor_slices((inputs['train'],targets['train'])).shuffle(len(targets['train']),seed=config['seed']).batch(32).with_options(options)
        validation=tf.data.Dataset.from_tensor_slices((inputs['validation'],targets['validation'])).batch(32).with_options(options)
        checkpoint=folder/'best.weights.h5'
        model.steps_per_execution=64
        history=model.fit(training,validation_data=validation,epochs=100,verbose=0,shuffle=False,callbacks=[
            tf.keras.callbacks.ModelCheckpoint(checkpoint,monitor='val_mae',mode='min',save_best_only=True,save_weights_only=True),
            tf.keras.callbacks.TerminateOnNaN(),
        ])
        curves=pd.DataFrame(history.history);curves.index=np.arange(1,len(curves)+1);curves.index.name='epoch'
        curves.to_csv(folder/'history.csv')
        fig,ax=plt.subplots(figsize=(7,4))
        ax.plot(curves.index,curves.loss,label='Training loss');ax.plot(curves.index,curves.val_loss,label='Validation loss')
        ax.set(xlabel='Epoch',ylabel=f"{config['loss'].upper()} ({'minutes squared' if config['loss']=='mse' else 'minutes'})",title=f"{config['architecture'].upper()} | {config['loss'].upper()} | {config['optimizer']} lr={config['learning_rate']}")
        ax.legend();fig.tight_layout();fig.savefig(folder/'loss.png',dpi=140);plt.close(fig)
        if len(curves)!=100 or not np.isfinite(curves[['loss','val_loss']].to_numpy()).all():
            raise ValueError('Numerical failure: this run did not complete 100 finite epochs')
        model.load_weights(checkpoint)
        prediction=model.predict(validation,verbose=0).reshape(-1)
        result.update(status='complete',epochs_completed=100,best_epoch=int(curves.val_mae.idxmin()),final_epoch_validation_mae_minutes=float(curves.val_mae.iloc[-1]),
                      **{'validation_'+k:v for k,v in metrics(targets['validation'],prediction).items()})
        model.save(folder/'best.keras')
        np.save(folder/'validation_predictions.npy',prediction)
    except Exception as exc:
        result.update(status='failed',error=f'{type(exc).__name__}: {exc}',epochs_completed=0 if history is None else len(history.epoch))
    result['seconds']=time.perf_counter()-start
    result_path.write_text(json.dumps(result,indent=2))
    return result



def collect_results(root: Path) -> pd.DataFrame:
    """Collect genuine logged run results; absent experiments are never fabricated."""
    result=pd.DataFrame([json.loads(p.read_text()) for p in (root/'outputs/runs').glob('*/result.json')])
    if not result.empty:
        if 'validation_mae_minutes' in result:result=result.sort_values('validation_mae_minutes')
        result.to_csv(root/'reports/experiment_results.csv',index=False)
    return result

In [ ]:
configurations = experiment_grid()
display(pd.DataFrame(configurations)[["architecture", "loss", "optimizer", "learning_rate", "epochs", "batch_size"]])
for architecture in ["mlp", "linear", "dnn"]:
    configuration = dict(architecture=architecture, loss="mae", optimizer="adam", learning_rate=0.001,
                         with_weather=True, epochs=100, batch_size=32, seed=SEED)
    model = build_tensorflow(configuration, inputs["train"].shape[1])
    model.summary()

## 5. Training experiments and loss curves
The full grid contains **36 configurations**. Each fit saves the training and validation loss for every epoch. Model comparisons use validation MAE in minutes, since MSE and MAE loss values have different units. Numerical failures, if any, are recorded separately and are not counted as completed 100-epoch runs.

In [ ]:
experiment_records = []
for configuration in configurations:
    record = fit_tensorflow(configuration, inputs, targets, ROOT, data_fingerprint)
    experiment_records.append(record)
    display(pd.DataFrame([{key: record.get(key) for key in ["architecture", "loss", "optimizer", "learning_rate",
                                                          "status", "epochs_completed", "best_epoch", "validation_mae_minutes"]}]))
    curve_path = ROOT / "outputs/runs" / record["run_id"] / "loss.png"
    if curve_path.exists():
        display(Image(filename=str(curve_path)))
results = collect_results(ROOT)
display(results[["architecture", "loss", "optimizer", "learning_rate", "status", "epochs_completed",
                 "best_epoch", "validation_mae_minutes", "validation_rmse_minutes", "validation_r2"]])

## 6. Best regression model and parameters
The best completed configuration for each architecture is compared using the same validation set. The overall model is selected by the lowest validation MAE. The table reports the selected checkpoint and the final epoch so overfitting remains visible.

In [ ]:
completed = results.loc[results["status"].eq("complete")].copy()
assert completed["epochs_completed"].eq(100).all()
best_by_model = completed.sort_values("validation_mae_minutes").groupby("architecture", as_index=False).first()
display(best_by_model[["architecture", "loss", "optimizer", "learning_rate", "epochs_completed", "batch_size",
                       "best_epoch", "validation_mae_minutes", "validation_rmse_minutes", "validation_r2",
                       "final_epoch_validation_mae_minutes"]])
winner = completed.sort_values("validation_mae_minutes").iloc[0].to_dict()
display(pd.Series(winner, name="selected_parameters").to_frame())

## 7. Predictions and loss review
The selected model is loaded and predictions are generated using **Keras `model.predict()`**. MSE, MAE, RMSE and R-squared are calculated on the validation set. The plots show predicted versus actual duration and residuals. Numerical metrics include every validation row; any focused display limits are labeled.

In [ ]:
def save_error_plots(actual: np.ndarray, predicted: np.ndarray, folder: Path, prefix: str) -> None:
    """Save actual/predicted and residual plots; any focused axes are explicitly labeled."""
    import matplotlib.pyplot as plt
    actual=np.asarray(actual).reshape(-1);predicted=np.asarray(predicted).reshape(-1)
    residual=predicted-actual
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    limit=max(1.,float(np.quantile(actual,.99)))
    axes[0].scatter(actual,predicted,s=7,alpha=.15)
    axes[0].plot([0,limit],[0,limit],color='black',linestyle='--')
    axes[0].set(xlim=(0,limit),ylim=(0,limit),xlabel='Actual minutes',ylabel='Predicted minutes',title='Focused display: actual-duration 99th percentile')
    axes[1].hist(residual,bins=80)
    axes[1].set(xlabel='Prediction minus actual (minutes)',ylabel='Trips',title='Residuals: all evaluation rows')
    fig.suptitle('Display limits do not remove rows from the error metrics')
    fig.tight_layout();folder.mkdir(parents=True,exist_ok=True);fig.savefig(folder/f'{prefix}_errors.png',dpi=140);plt.close(fig)

In [ ]:
best_model = tf.keras.models.load_model(ROOT / "outputs/runs" / winner["run_id"] / "best.keras")
prediction_dataset = tf.data.Dataset.from_tensor_slices(inputs["validation"]).batch(32)
options = tf.data.Options()
options.threading.private_threadpool_size = 1
prediction_dataset = prediction_dataset.with_options(options)
predictions = best_model.predict(prediction_dataset, verbose=0).reshape(-1)
prediction_metrics = metrics(targets["validation"], predictions)
prediction_metrics["mse_minutes_squared"] = float(np.mean((targets["validation"] - predictions) ** 2))
display(pd.Series(prediction_metrics, name="prediction_metrics").to_frame())
prediction_table = splits["validation"][["row_id", "duration_minutes"]].copy()
prediction_table["predicted_minutes"] = predictions
prediction_table["absolute_error_minutes"] = np.abs(prediction_table["duration_minutes"] - predictions)
display(prediction_table.sample(n=10, random_state=SEED))
save_error_plots(targets["validation"], predictions, ROOT / "outputs", "validation")
display(Image(filename=str(ROOT / "outputs/validation_errors.png")))
(ROOT / "reports/selected_model.json").write_text(json.dumps(winner, indent=2))
(ROOT / "reports/prediction_metrics.json").write_text(json.dumps(prediction_metrics, indent=2))
prediction_table.to_csv(ROOT / "outputs/validation_predictions.csv", index=False)

## 8. Outliers and overfitting
MSE gives large errors more influence than MAE. Unusually long recorded trips remain in the sampled labels, so RMSE and R-squared must be reviewed alongside MAE. Training and validation curves are used to identify overfitting; selecting a validation checkpoint does not shorten the required 100-epoch training run.

In [ ]:
overfitting = completed[["architecture", "loss", "optimizer", "learning_rate", "best_epoch",
                          "validation_mae_minutes", "final_epoch_validation_mae_minutes"]].copy()
overfitting["mae_increase_after_best_checkpoint"] = overfitting["final_epoch_validation_mae_minutes"] - overfitting["validation_mae_minutes"]
display(overfitting.sort_values("validation_mae_minutes").head(6))
display(prediction_table.nlargest(5, "absolute_error_minutes"))

## 9. Extra credit: PyTorch DNN and framework comparison
The best TensorFlow DNN configuration is rebuilt in PyTorch with two hidden layers, the same input arrays, split, scaling, optimizer, learning rate, loss, 100 epochs and batch size 32. Each framework retains its best validation-MAE checkpoint. Initialization and backend operations differ, so matching seeds do not imply identical predictions.

In [ ]:
def fit_pytorch(config: dict, inputs: dict[str,np.ndarray], targets: dict[str,np.ndarray], root: Path) -> dict:
    """Train the extra-credit 64/32 DNN for 100 epochs with the same input arrays.

    Save the validation-MAE checkpoint and training/validation loss. Test arrays
    are not accessed. Matching seeds do not imply identical framework results.
    """
    import torch
    import matplotlib.pyplot as plt
    config={k:(v.item() if isinstance(v,np.generic) else v) for k,v in config.items()}
    assert config['architecture']=='dnn' and config['epochs']==100 and config['batch_size']==32
    torch.manual_seed(config['seed']);torch.set_num_threads(2)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=torch.nn.Sequential(torch.nn.Linear(inputs['train'].shape[1],64),torch.nn.ReLU(),torch.nn.Linear(64,32),torch.nn.ReLU(),torch.nn.Linear(32,1)).to(device)
    optimizer={'sgd':torch.optim.SGD,'adam':torch.optim.Adam,'rmsprop':torch.optim.RMSprop}[config['optimizer']](model.parameters(),lr=config['learning_rate'])
    # RMSprop defaults differ: match Keras rho=.9 and epsilon=1e-7; Adam epsilon too.
    if config['optimizer']=='rmsprop':optimizer=torch.optim.RMSprop(model.parameters(),lr=config['learning_rate'],alpha=.9,eps=1e-7)
    elif config['optimizer']=='adam':optimizer=torch.optim.Adam(model.parameters(),lr=config['learning_rate'],eps=1e-7)
    loss_fn=torch.nn.MSELoss() if config['loss']=='mse' else torch.nn.L1Loss()
    x=torch.from_numpy(inputs['train']);y=torch.from_numpy(targets['train']).reshape(-1,1)
    xv=torch.from_numpy(inputs['validation']).to(device);yv=torch.from_numpy(targets['validation']).reshape(-1,1).to(device)
    loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x,y),batch_size=32,shuffle=True,generator=torch.Generator().manual_seed(config['seed']))
    folder=root/'outputs/pytorch';folder.mkdir(exist_ok=True)
    best=float('inf');history=[];start=time.perf_counter()
    for epoch in range(100):
        model.train();total=0.
        for xb,yb in loader:
            xb=xb.to(device);yb=yb.to(device)
            optimizer.zero_grad();prediction=model(xb);loss=loss_fn(prediction,yb)
            if not torch.isfinite(loss):raise ValueError('PyTorch numerical failure; run incomplete')
            loss.backward();optimizer.step();total+=float(loss.detach())*len(xb)
        model.eval()
        with torch.no_grad():
            prediction=model(xv);val_loss=float(loss_fn(prediction,yv));val_mae=float(torch.mean(torch.abs(prediction-yv)))
        history.append({'epoch':epoch+1,'loss':total/len(y),'val_loss':val_loss,'val_mae':val_mae})
        if val_mae<best:best=val_mae;torch.save(model.state_dict(),folder/'best.pt')
    model.load_state_dict(torch.load(folder/'best.pt',weights_only=True,map_location=device));model.eval()
    with torch.no_grad():prediction=model(xv).cpu().numpy().reshape(-1)
    curves=pd.DataFrame(history);curves.to_csv(folder/'history.csv',index=False)
    fig,ax=plt.subplots(figsize=(7,4));ax.plot(curves.epoch,curves.loss,label='Training loss');ax.plot(curves.epoch,curves.val_loss,label='Validation loss');ax.legend();ax.set(xlabel='Epoch',ylabel=config['loss'].upper(),title='PyTorch extra-credit DNN');fig.tight_layout();fig.savefig(folder/'loss.png',dpi=140);plt.close(fig)
    result={'framework':'pytorch','hardware':str(device),**config,'epochs_completed':100,'seconds':time.perf_counter()-start,**metrics(targets['validation'],prediction)}
    (folder/'result.json').write_text(json.dumps(result,indent=2))
    return result

In [ ]:
tensorflow_dnn = completed.loc[completed["architecture"].eq("dnn")].sort_values("validation_mae_minutes").iloc[0].to_dict()
pytorch_configuration = {key: tensorflow_dnn[key] for key in configurations[0]}
pytorch_path = ROOT / "outputs/pytorch/result.json"
if pytorch_path.exists():
    pytorch_result = json.loads(pytorch_path.read_text())
    assert all(pytorch_result[key] == value for key, value in pytorch_configuration.items())
    assert pytorch_result["epochs_completed"] == 100
else:
    pytorch_result = fit_pytorch(pytorch_configuration, inputs, targets, ROOT)
framework_comparison = pd.DataFrame([
    {"framework": "TensorFlow", "architecture": "64 → 32 → 1", "loss": tensorflow_dnn["loss"],
     "optimizer": tensorflow_dnn["optimizer"], "learning_rate": tensorflow_dnn["learning_rate"],
     "epochs": 100, "batch_size": 32, "mae_minutes": tensorflow_dnn["validation_mae_minutes"],
     "rmse_minutes": tensorflow_dnn["validation_rmse_minutes"], "r2": tensorflow_dnn["validation_r2"]},
    {"framework": "PyTorch", "architecture": "64 → 32 → 1", "loss": pytorch_result["loss"],
     "optimizer": pytorch_result["optimizer"], "learning_rate": pytorch_result["learning_rate"],
     "epochs": 100, "batch_size": 32, "mae_minutes": pytorch_result["mae_minutes"],
     "rmse_minutes": pytorch_result["rmse_minutes"], "r2": pytorch_result["r2"]},
])
display(framework_comparison)
display(Image(filename=str(ROOT / "outputs/pytorch/loss.png")))
framework_comparison.to_csv(ROOT / "reports/framework_comparison.csv", index=False)

## 10. Results and conclusion
The final numerical findings and framework comparison are reported below from the completed experiments.